# Обработка экспериментальных данных с использованием метода наименьших квадратов (МНК)

## 1. Введение
В реальных физических экспериментах данные всегда содержат погрешность (шум), вызванную неточностью измерительных приборов или внешними факторами. Задача исследователя — найти математическую закономерность, которая описывает эти данные наилучшим образом.
Метод наименьших квадратов (МНК) — это математический метод, применяемый для решения различных задач, основанный на минимизации суммы квадратов отклонений заданных функций от искомых.
В данной работе мы предполагается, что экспериментальные данные описываются процессом с ускорением, поэтому для аппроксимации выбран полином второй степени (парабола): y = β0 + β1x + β2x^2
Где β0, β1, β2 — неизвестные коэффициенты, которые необходимо найти.

In [ ]:
# Подключение необходимых библиотек
import sqlite3
import numpy as np
import tkinter as tk
from tkinter import messagebox
from matplotlib.figure import Figure
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg

## 2. Подготовка и генерация данных
Для имитации реального эксперимента, необходимо сгенерировать "идеальные" значения функции y = 0.5x^2 - 2x + 10 и добавить к ним случайный шум.
Для обеспечения сохранности данных и разделения этапов сбора и обработки, полученные точки сохраняются в локальную базу данных SQLite.

In [ ]:
# Подготовка БД и генерация данных, примерно подходящих для параболы
def init_db():
    conn = sqlite3.connect('experiment.db')
    cursor = conn.cursor()
    cursor.execute('''CREATE TABLE IF NOT EXISTS points (x REAL, y REAL)''')
    cursor.execute('DELETE FROM points')
    conn.commit()
    return conn, cursor

def generate_data():
    conn, cursor = init_db()
    x = np.linspace(-10, 10, 100)
    # начальная функция y = 0.5*x^2 - 2*x + 10
    # добавляем случайный шум с нормальным распределением (среднее 0, отклонение 5)
    noise = np.random.normal(0, 5, 100)
    y = 0.5 * x**2 - 2 * x + 10 + noise
    # сохраняем в базу данных
    for i in range(len(x)):
        cursor.execute('INSERT INTO points (x, y) VALUES (?, ?)', (x[i], y[i]))
    
    conn.commit()
    conn.close()
    return x, y

def load_data():
    conn = sqlite3.connect('experiment.db')
    cursor = conn.cursor()
    cursor.execute('SELECT x, y FROM points')
    rows = cursor.fetchall()
    conn.close()
    
    if not rows:
        return np.array([]), np.array([])
        
    x = np.array([row[0] for row in rows])
    y = np.array([row[1] for row in rows])
    return x, y

def add_single_point(x_val, y_val):
    # добавить одну точку в базу данных (для ручного ввода)
    conn = sqlite3.connect('experiment.db')
    cursor = conn.cursor()
    cursor.execute('INSERT INTO points (x, y) VALUES (?, ?)', (x_val, y_val))
    conn.commit()
    conn.close()

## 3. Своя реализация и встроенная функция
В матричном виде задача нахождения коэффициентов полинома сводится к решению переопределенной системы уравнений. Вектор коэффициентов β находится по формуле:
β = (A^T A)^(-1) A^T y,

Где A — матрица признаков, столбцы которой состоят из единиц (для свободного члена), значений x и x^2. В коде данный алгоритм реализуется вручную средствами линейной алгебры numpy, а также стандартной функцией numpy.polyfit для проверки.

In [ ]:
# МНК полинома второй степени
def custom_ols(x, y):
    # создаем матрицу признаков A: столбцы [1, x, x^2]
    A = np.vstack([np.ones(len(x)), x, x**2]).T
    # beta = (A^T * A)^-1 * A^T * y
    A_T = A.T
    beta = np.linalg.inv(A_T.dot(A)).dot(A_T).dot(y)
    # [b0, b1, b2] для уравнения y = b0 + b1*x + b2*x^2
    return beta

def numpy_ols(x, y):
    # [b2, b1, b0]
    beta_np = np.polyfit(x, y, 2)
    return beta_np

def calculate_y(x, beta, is_numpy=False):
    if is_numpy:
        # [b2, b1, b0]
        return beta[0]*x**2 + beta[1]*x + beta[2]
    else:
        # [b0, b1, b2]
        return beta[0] + beta[1]*x + beta[2]*x**2

## 4. Графический интерфейс и визуализация результатов
Для удобства взаимодействия с программой разработан графический интерфейс с использованием библиотеки tkinter. Внутри окна размещен график matplotlib, который позволяет визуально оценить качество аппроксимации.

In [ ]:
# Графический интерфейс и визуализация
class App:
    def __init__(self, root):
        self.root = root
        self.root.title("Обработка экспериментальных данных: МНК")
        
        self.x_data = np.array([])
        self.y_data = np.array([])

        control_frame = tk.Frame(root)
        control_frame.pack(side=tk.TOP, fill=tk.X, padx=10, pady=5)
        
        tk.Button(control_frame, text="Сгенерировать БД", command=self.gui_generate).pack(side=tk.LEFT, padx=5)
        tk.Button(control_frame, text="Загрузить из БД", command=self.gui_load).pack(side=tk.LEFT, padx=5)
        tk.Button(control_frame, text="Аппроксимировать", command=self.gui_approximate, bg="#00AAAA").pack(side=tk.LEFT, padx=5)

        # Ручной ввод
        manual_frame = tk.Frame(root)
        manual_frame.pack(side=tk.TOP, fill=tk.X, padx=10, pady=5)
        tk.Label(manual_frame, text="Ручной ввод -> X:").pack(side=tk.LEFT)
        self.entry_x = tk.Entry(manual_frame, width=8)
        self.entry_x.pack(side=tk.LEFT, padx=5)
        tk.Label(manual_frame, text="Y:").pack(side=tk.LEFT)
        self.entry_y = tk.Entry(manual_frame, width=8)
        self.entry_y.pack(side=tk.LEFT, padx=5)
        tk.Button(manual_frame, text="Добавить точку", command=self.gui_add_point).pack(side=tk.LEFT, padx=5)

        self.info_text = tk.Text(root, height=4, font=("Consolas", 10))
        self.info_text.pack(side=tk.TOP, fill=tk.X, padx=10, pady=5)
        self.info_text.insert(tk.END, "Сгенерируйте или загрузите данные.\n")

        self.fig = Figure(figsize=(8, 5), dpi=100)
        self.ax = self.fig.add_subplot(111)
        self.canvas = FigureCanvasTkAgg(self.fig, master=root)
        self.canvas.get_tk_widget().pack(side=tk.BOTTOM, fill=tk.BOTH, expand=True)

    def gui_generate(self):
        self.x_data, self.y_data = generate_data()
        self.info_text.delete(1.0, tk.END)
        self.info_text.insert(tk.END, "Сгенерировано 100 точек с шумом и сохранено в БД.\n")

    def gui_load(self):
        self.x_data, self.y_data = load_data()
        self.info_text.delete(1.0, tk.END)
        self.info_text.insert(tk.END, f"Загружено {len(self.x_data)} точек из БД.\n")
        self.plot_raw_data()

    def gui_add_point(self):
        try:
            x_val = float(self.entry_x.get())
            y_val = float(self.entry_y.get())
            add_single_point(x_val, y_val)
            self.entry_x.delete(0, tk.END)
            self.entry_y.delete(0, tk.END)
            self.gui_load() # Перезагружаем график с новой точкой
        except ValueError:
            messagebox.showerror("Ошибка", "Координаты должны быть числами!")

    def plot_raw_data(self):
        self.ax.clear()
        self.ax.scatter(self.x_data, self.y_data, color='blue', label='Экспериментальные данные', s=15, alpha=0.6)
        self.ax.grid(True, linestyle='--', alpha=0.5)
        self.ax.legend()
        self.canvas.draw()

    def gui_approximate(self):
        if len(self.x_data) < 3:
            messagebox.showwarning("Внимание", "Для параболы нужно минимум 3 точки!")
            return

        beta_custom = custom_ols(self.x_data, self.y_data)
        beta_np = numpy_ols(self.x_data, self.y_data)

        self.info_text.delete(1.0, tk.END)
        self.info_text.insert(tk.END, f"Свой МНК: y = {beta_custom[0]:.6f} + {beta_custom[1]:.6f}x + {beta_custom[2]:.6f}x^2\n")
        self.info_text.insert(tk.END, f"Встроенный Numpy: y = {beta_np[2]:.6f} + {beta_np[1]:.6f}x + {beta_np[0]:.6f}x^2\n")

        self.ax.clear()
        self.ax.scatter(self.x_data, self.y_data, color='blue', label='Экспериментальные данные', s=15, alpha=0.6)
        # плавная ось x
        x_line = np.linspace(min(self.x_data), max(self.x_data), 200)
        
        y_np = calculate_y(x_line, beta_np, is_numpy=True)
        y_custom = calculate_y(x_line, beta_custom, is_numpy=False)

        self.ax.plot(x_line, y_np, color='red', label='numpy.polyfit', linewidth=4)
        self.ax.plot(x_line, y_custom, color='lightgreen', linestyle='--', label='Своя реализация МНК', linewidth=2)

        self.ax.grid(True, linestyle='--', alpha=0.5)
        self.ax.legend()
        self.canvas.draw()

# Запуск приложения
if __name__ == "__main__":
    root = tk.Tk()
    app = App(root)
    root.mainloop()

## Заключение

В ходе работы была успешно продемонстрирована математическая модель МНК. Сравнение результатов собственной матричной реализации алгоритма со стандартной функцией numpy.polyfit показало полное совпадение коэффициентов аппроксимации. Это подтверждает корректность работы реализованного математического алгоритма программы.